# Module 5 - Defensive Federated Learning

Module 4 showed that malicious clients can damage naive FedAvg. Module 5 asks whether the server can reduce that damage without knowing which clients are malicious.

The server-side idea is simple: keep the same attack recipe, but stop blindly averaging every selected update.

## 1. Setup

This notebook can be run from the repository root or from inside `5_Defensive_FL/`.

In [ ]:
from pathlib import Path
import json
import sys

import yaml

cwd = Path.cwd()
MODULE_DIR = cwd if cwd.name == "5_Defensive_FL" else cwd / "5_Defensive_FL"
REPO_ROOT = MODULE_DIR.parent
MODULE4_DIR = REPO_ROOT / "4_Adversarial_FL"

for path in (MODULE_DIR, MODULE4_DIR, REPO_ROOT):
    path_str = str(path.resolve())
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from defensive_servers import make_attack_config, run_defensive_fl, validate_module5_config
from metrics import build_comparison_rows, save_csv, save_json, update_norm_rows
from plots import (
    plot_accuracy_curves,
    plot_asr_curves,
    plot_defense_comparison,
    plot_sweep_metric,
    plot_update_norm_histogram,
)

ARTIFACT_DIR = MODULE_DIR / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = MODULE_DIR / "config.yaml"
with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

issues = validate_module5_config(config, require_attack=True)
if issues:
    raise ValueError("Config validation failed:\n- " + "\n- ".join(issues))

config

## 2. Threat Model Recap

The attacker controls some clients. The attacker can poison local training batches through Module 4's `MaliciousClient`. The attacker cannot edit the server directly. The server receives client updates and must aggregate them without knowing which selected clients are malicious.

In [ ]:
global_config = config["global_config"]
data_config = config["data_config"]
fed_config = config["fed_config"]
model_config = config["model_config"]
optim_config = config.get("optim_config", {})
base_attack_config = config["attack"]


def artifact_path(name):
    return ARTIFACT_DIR / name


def load_json_if_present(path):
    if not path.exists():
        return None
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def run_module5_experiment(run_name, attack_config, defense_config):
    server = run_defensive_fl(
        global_config=global_config,
        data_config=data_config,
        fed_config=fed_config,
        model_config=model_config,
        optim_config=optim_config,
        attack_config=attack_config,
        defense_config=defense_config,
    )
    result = server.results
    save_json(result, artifact_path(f"module5_{run_name}.json"))
    return result

## 3. Baselines

Run a clean FedAvg baseline and an attacked FedAvg baseline before introducing defenses. Set `RUN_BASELINES = True` to execute the runs. Leave it `False` to load prior artifacts.

In [ ]:
RUN_BASELINES = False

run_results = {}

if RUN_BASELINES:
    clean_attack = make_attack_config(
        base_attack_config,
        malicious_fraction=0.0,
        poison_rate=0.0,
    )
    run_results["clean_fedavg"] = run_module5_experiment(
        "clean_fedavg",
        clean_attack,
        {"name": "fedavg"},
    )

    run_results["attacked_fedavg"] = run_module5_experiment(
        "attacked_fedavg",
        base_attack_config,
        {"name": "fedavg"},
    )
else:
    for run_name in ("clean_fedavg", "attacked_fedavg"):
        loaded = load_json_if_present(artifact_path(f"module5_{run_name}.json"))
        if loaded is not None:
            run_results[run_name] = loaded

run_results.keys()

## 4. Update Diagnostics

Before comparing defenses, inspect why FedAvg is vulnerable. Update norm, cosine-to-mean, and distance-to-median diagnostics are stored for every round.

In [ ]:
if "attacked_fedavg" in run_results:
    norm_rows = update_norm_rows(run_results["attacked_fedavg"])
    save_json(norm_rows, artifact_path("module5_update_diagnostics.json"))
    plot_update_norm_histogram(
        norm_rows,
        artifact_path("module5_update_norms.png"),
        round_number=base_attack_config["start_round"],
    )
else:
    print("Run or load attacked_fedavg before plotting update diagnostics.")

## 5. Defense Comparison

Run each defense under the same Module 4 attack recipe. Set `RUN_DEFENSE_COMPARISON = True` to execute the full comparison.

In [ ]:
RUN_DEFENSE_COMPARISON = False

if RUN_DEFENSE_COMPARISON:
    for defense_config in config["experiments"]["defenses"]:
        name = defense_config["name"]
        if name == "fedavg" and "attacked_fedavg" in run_results:
            continue
        run_results[name] = run_module5_experiment(
            name,
            base_attack_config,
            defense_config,
        )
else:
    for defense_config in config["experiments"]["defenses"]:
        name = "attacked_fedavg" if defense_config["name"] == "fedavg" else defense_config["name"]
        loaded = load_json_if_present(artifact_path(f"module5_{name}.json"))
        if loaded is not None:
            run_results[name] = loaded

comparison_rows = build_comparison_rows(run_results)
if comparison_rows:
    save_json(comparison_rows, artifact_path("module5_defense_comparison.json"))
    save_csv(comparison_rows, artifact_path("module5_defense_comparison.csv"))
else:
    print("No defense comparison rows yet. Run baselines or load prior artifacts first.")
comparison_rows

In [ ]:
if comparison_rows:
    plot_accuracy_curves(
        run_results,
        artifact_path("module5_accuracy_curves.png"),
        attack_start_round=base_attack_config["start_round"],
    )
    plot_asr_curves(
        run_results,
        artifact_path("module5_asr_curves.png"),
        attack_start_round=base_attack_config["start_round"],
    )
    plot_defense_comparison(
        comparison_rows,
        metric="final_accuracy",
        path=artifact_path("module5_accuracy_vs_defense.png"),
        ylabel="Final accuracy (%)",
        title="Final accuracy by defense",
    )
    plot_defense_comparison(
        comparison_rows,
        metric="asr",
        path=artifact_path("module5_asr_vs_defense.png"),
        ylabel="Final ASR (%)",
        title="Final ASR by defense",
    )
else:
    print("No comparison rows available yet.")

## 6. Malicious Fraction Sweep

This sweep shows where each defense starts to break as the malicious-client fraction increases.

In [ ]:
RUN_MALICIOUS_FRACTION_SWEEP = False

sweep_rows = []
if RUN_MALICIOUS_FRACTION_SWEEP:
    for malicious_fraction in config["experiments"]["malicious_fraction_sweep"]:
        attack_config = make_attack_config(base_attack_config, malicious_fraction=malicious_fraction)
        for defense_config in config["experiments"]["defenses"]:
            run_name = f"{defense_config['name']}_mf_{malicious_fraction}"
            result = run_module5_experiment(run_name, attack_config, defense_config)
            row = build_comparison_rows({run_name: result})[0]
            row["defense"] = defense_config["name"]
            row["malicious_fraction"] = malicious_fraction
            sweep_rows.append(row)

    save_json(sweep_rows, artifact_path("module5_malicious_fraction_sweep.json"))
    plot_sweep_metric(
        sweep_rows,
        x_key="malicious_fraction",
        y_key="final_accuracy",
        group_key="defense",
        path=artifact_path("module5_malicious_fraction_accuracy.png"),
        ylabel="Final accuracy (%)",
    )
    plot_sweep_metric(
        sweep_rows,
        x_key="malicious_fraction",
        y_key="asr",
        group_key="defense",
        path=artifact_path("module5_malicious_fraction_asr.png"),
        ylabel="Final ASR (%)",
    )
else:
    sweep_rows = load_json_if_present(artifact_path("module5_malicious_fraction_sweep.json")) or []

sweep_rows[:3]

## 7. Non-IID Stress Test

Robust aggregation is harder when honest clients already look different. This test keeps the attack recipe fixed and increases `non_iid_per`.

In [ ]:
RUN_NON_IID_STRESS = False

non_iid_rows = []
if RUN_NON_IID_STRESS:
    selected_defenses = [
        {"name": "fedavg"},
        {"name": "clipping", "clip_norm": 5.0},
        {"name": "trimmed_mean", "trim_fraction": 0.1},
        {"name": "krum", "byzantine_f": 2},
    ]
    for non_iid_per in config["experiments"]["non_iid_sweep"]:
        data_variant = dict(data_config)
        data_variant["non_iid_per"] = non_iid_per
        for defense_config in selected_defenses:
            server = run_defensive_fl(
                global_config=global_config,
                data_config=data_variant,
                fed_config=fed_config,
                model_config=model_config,
                optim_config=optim_config,
                attack_config=base_attack_config,
                defense_config=defense_config,
            )
            run_name = f"{defense_config['name']}_noniid_{non_iid_per}"
            row = build_comparison_rows({run_name: server.results})[0]
            row["defense"] = defense_config["name"]
            row["non_iid_per"] = non_iid_per
            non_iid_rows.append(row)

    save_json(non_iid_rows, artifact_path("module5_non_iid_defense_stress.json"))
    plot_sweep_metric(
        non_iid_rows,
        x_key="non_iid_per",
        y_key="final_accuracy",
        group_key="defense",
        path=artifact_path("module5_non_iid_accuracy.png"),
        ylabel="Final accuracy (%)",
    )
    plot_sweep_metric(
        non_iid_rows,
        x_key="non_iid_per",
        y_key="asr",
        group_key="defense",
        path=artifact_path("module5_non_iid_asr.png"),
        ylabel="Final ASR (%)",
    )
else:
    non_iid_rows = load_json_if_present(artifact_path("module5_non_iid_defense_stress.json")) or []

non_iid_rows[:3]

## 8. Discussion

Use the saved tables and plots to answer:

1. Which defense preserved clean accuracy best?
2. Which defense reduced ASR the most?
3. Which defense failed under non-IID data?
4. Did clipping help or hurt?
5. Did Krum select reasonable updates?
6. What assumption does each defense make?

Workshop synthesis:

- Module 1: FedAvg works when clients are simple and honest.
- Module 2: Non-IID data makes honest FL harder.
- Module 3: Optimizers can improve convergence.
- Module 4: Malicious clients can poison FL.
- Module 5: Robust aggregation can help, but every defense has assumptions and tradeoffs.